# OpenDataCopilot - Exploration des Données

**Projet Master 2 Data Science**

Ce notebook explore les données téléchargées pour le projet OpenDataCopilot :
- **Santé publique** : Hospitalisations COVID-19, urgences, démographie médicale
- **Pollution** : Indices Airparif, mesures OpenAQ (Paris, Lyon, Marseille)

---

## Objectifs
1. Comprendre la structure et le contenu de chaque dataset
2. Identifier les colonnes clés pour le RAG
3. Détecter les problèmes de qualité des données
4. Visualiser les tendances et patterns
5. Préparer les données pour l'indexation vectorielle

---
# SECTION 1 - Configuration & Chargement des Données
---

In [ ]:
# Imports
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import warnings
import os

# Visualisation
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Configuration
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

# ═══════════════════════════════════════════════════════════
# Détection robuste du répertoire projet
# ═══════════════════════════════════════════════════════════
def find_project_root() -> Path:
    """Trouve la racine du projet en cherchant des fichiers marqueurs."""
    # Méthode 1: Chercher à partir du répertoire courant
    current = Path.cwd()
    
    # Fichiers qui indiquent la racine du projet
    markers = ['pyproject.toml', 'requirements.txt', '.git', 'README.md']
    
    # Remonter jusqu'à 5 niveaux
    for _ in range(5):
        if any((current / marker).exists() for marker in markers):
            return current
        current = current.parent
    
    # Méthode 2: Chemin absolu hardcodé en fallback
    fallback = Path("/Users/jerome/Documents/University/Master2/PROJET /Open_Data_Copilot")
    if fallback.exists():
        return fallback
    
    # Dernier recours
    return Path.cwd()

PROJECT_ROOT = find_project_root()
DATA_SANTE = PROJECT_ROOT / 'data' / 'raw' / 'sante'
DATA_POLLUTION = PROJECT_ROOT / 'data' / 'raw' / 'pollution'

print(f"📁 Racine projet: {PROJECT_ROOT}")
print(f"📁 Répertoire santé: {DATA_SANTE}")
print(f"📁 Répertoire pollution: {DATA_POLLUTION}")
print(f"   → Existe: {DATA_SANTE.exists()}")
print(f"   → Existe: {DATA_POLLUTION.exists()}")

In [ ]:
# Fonction de chargement sécurisé
def load_csv_safe(filepath: Path, **kwargs) -> pd.DataFrame | None:
    """Charge un CSV avec gestion d'erreurs."""
    try:
        df = pd.read_csv(filepath, **kwargs)
        print(f"✅ {filepath.name}: {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
        return df
    except FileNotFoundError:
        print(f"❌ Fichier non trouvé: {filepath}")
        return None
    except Exception as e:
        print(f"❌ Erreur chargement {filepath.name}: {e}")
        return None

# Dictionnaire pour stocker tous les datasets
datasets = {}

In [ ]:
# ═══════════════════════════════════════════════════════════
# Chargement des données SANTÉ
# ═══════════════════════════════════════════════════════════
print("\n" + "="*60)
print("📊 DONNÉES SANTÉ PUBLIQUE")
print("="*60 + "\n")

# Hospitalisations COVID-19
datasets['covid_hosp'] = load_csv_safe(
    DATA_SANTE / 'covid_hospitalisations.csv',
    sep=';',
    low_memory=False
)

# Urgences SurSaUD
datasets['urgences'] = load_csv_safe(
    DATA_SANTE / 'sursaud_urgences.csv',
    sep=';',
    low_memory=False
)

# Tests COVID
datasets['covid_tests'] = load_csv_safe(
    DATA_SANTE / 'covid_tests_dep.csv',
    sep=';'
)

# Professionnels de santé
datasets['medecins'] = load_csv_safe(
    DATA_SANTE / 'professionnels_sante_dep.csv',
    sep=','
)

In [ ]:
# ═══════════════════════════════════════════════════════════
# Chargement des données POLLUTION
# ═══════════════════════════════════════════════════════════
print("\n" + "="*60)
print("🌍 DONNÉES POLLUTION")
print("="*60 + "\n")

# Airparif
datasets['airparif'] = load_csv_safe(
    DATA_POLLUTION / 'airparif_indices.csv'
)

# OpenAQ par ville
datasets['openaq_paris'] = load_csv_safe(
    DATA_POLLUTION / 'openaq_paris_latest.csv'
)
datasets['openaq_lyon'] = load_csv_safe(
    DATA_POLLUTION / 'openaq_lyon_latest.csv'
)
datasets['openaq_marseille'] = load_csv_safe(
    DATA_POLLUTION / 'openaq_marseille_latest.csv'
)

In [ ]:
# Résumé des datasets chargés
print("\n" + "="*60)
print("📋 RÉSUMÉ DES DATASETS")
print("="*60)

summary_data = []
for name, df in datasets.items():
    if df is not None:
        memory_mb = df.memory_usage(deep=True).sum() / 1024 / 1024
        summary_data.append({
            'Dataset': name,
            'Lignes': f"{df.shape[0]:,}",
            'Colonnes': df.shape[1],
            'Mémoire (MB)': f"{memory_mb:.2f}"
        })

summary_df = pd.DataFrame(summary_data)
display(summary_df)

### Aperçu des premières lignes de chaque dataset

In [ ]:
# Afficher les premières lignes de chaque dataset
for name, df in datasets.items():
    if df is not None:
        print(f"\n{'='*60}")
        print(f"📄 {name.upper()}")
        print(f"{'='*60}")
        display(df.head(3))
        print(f"\nColonnes: {list(df.columns)}")

---
# SECTION 2 - Statistiques Descriptives
---

In [ ]:
def analyze_dataset(name: str, df: pd.DataFrame) -> dict:
    """Analyse complète d'un dataset."""
    if df is None:
        return None
    
    print(f"\n{'═'*70}")
    print(f"📊 ANALYSE: {name.upper()}")
    print(f"{'═'*70}")
    
    # Dimensions
    print(f"\n📐 Dimensions: {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
    
    # Types de données
    print(f"\n📋 Types de données:")
    for dtype, count in df.dtypes.value_counts().items():
        print(f"   {dtype}: {count} colonnes")
    
    # Colonnes temporelles (détection automatique)
    date_cols = [c for c in df.columns if any(x in c.lower() for x in ['date', 'jour', 'time', 'semaine'])]
    if date_cols:
        print(f"\n📅 Colonnes temporelles détectées: {date_cols}")
        for col in date_cols[:2]:  # Limiter à 2
            try:
                dates = pd.to_datetime(df[col], errors='coerce')
                if dates.notna().any():
                    print(f"   {col}: {dates.min()} → {dates.max()}")
            except:
                pass
    
    # Colonnes géographiques
    geo_cols = [c for c in df.columns if any(x in c.lower() for x in ['dep', 'reg', 'ville', 'city', 'location'])]
    if geo_cols:
        print(f"\n🗺️ Colonnes géographiques: {geo_cols}")
        for col in geo_cols[:2]:
            n_unique = df[col].nunique()
            print(f"   {col}: {n_unique} valeurs uniques")
    
    # Statistiques numériques
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        print(f"\n📈 Statistiques numériques:")
        display(df[numeric_cols].describe().round(2))
    
    return {
        'shape': df.shape,
        'date_cols': date_cols,
        'geo_cols': geo_cols,
        'numeric_cols': list(numeric_cols)
    }

In [ ]:
# Analyser tous les datasets
analyses = {}
for name, df in datasets.items():
    if df is not None:
        analyses[name] = analyze_dataset(name, df)

---
# SECTION 3 - Visualisations Santé
---

## 3.1 Évolution des hospitalisations COVID-19

In [ ]:
# Préparation des données COVID hospitalisations
if datasets['covid_hosp'] is not None:
    df_covid = datasets['covid_hosp'].copy()
    
    # Identifier la colonne de date
    date_col = None
    for col in ['jour', 'date', 'Date']:
        if col in df_covid.columns:
            date_col = col
            break
    
    if date_col:
        df_covid[date_col] = pd.to_datetime(df_covid[date_col], errors='coerce')
        print(f"Colonne de date utilisée: {date_col}")
        print(f"Plage: {df_covid[date_col].min()} → {df_covid[date_col].max()}")
    
    # Afficher les colonnes disponibles
    print(f"\nColonnes disponibles: {list(df_covid.columns)}")

In [ ]:
# Graphique 1: Évolution temporelle des hospitalisations France entière
if datasets['covid_hosp'] is not None and date_col:
    # Agréger par date (total France)
    hosp_col = None
    for col in ['hosp', 'hospitalisations', 'nb_hosp']:
        if col in df_covid.columns:
            hosp_col = col
            break
    
    if hosp_col:
        # Filtrer les données valides et agréger
        df_daily = df_covid.groupby(date_col)[hosp_col].sum().reset_index()
        df_daily = df_daily.dropna()
        
        fig = px.line(
            df_daily,
            x=date_col,
            y=hosp_col,
            title='📈 Évolution des hospitalisations COVID-19 en France',
            labels={date_col: 'Date', hosp_col: 'Nombre d\'hospitalisations'}
        )
        fig.update_layout(
            template='plotly_white',
            hovermode='x unified'
        )
        fig.show()
    else:
        print("Colonne d'hospitalisations non trouvée")

In [ ]:
# Graphique 2: Top 10 départements les plus touchés
if datasets['covid_hosp'] is not None and hosp_col:
    dep_col = None
    for col in ['dep', 'departement', 'code_dep']:
        if col in df_covid.columns:
            dep_col = col
            break
    
    if dep_col:
        # Total par département
        df_by_dep = df_covid.groupby(dep_col)[hosp_col].sum().sort_values(ascending=False).head(10)
        
        fig = px.bar(
            x=df_by_dep.index.astype(str),
            y=df_by_dep.values,
            title='🏥 Top 10 départements - Hospitalisations COVID-19 cumulées',
            labels={'x': 'Département', 'y': 'Hospitalisations cumulées'},
            color=df_by_dep.values,
            color_continuous_scale='Reds'
        )
        fig.update_layout(template='plotly_white', showlegend=False)
        fig.show()

## 3.2 Passages aux urgences (SurSaUD)

In [ ]:
# Analyse des urgences
if datasets['urgences'] is not None:
    df_urg = datasets['urgences'].copy()
    print(f"Colonnes urgences: {list(df_urg.columns)}")
    
    # Trouver la colonne de date
    date_col_urg = None
    for col in ['date_de_passage', 'date', 'jour']:
        if col in df_urg.columns:
            date_col_urg = col
            df_urg[col] = pd.to_datetime(df_urg[col], errors='coerce')
            break
    
    # Trouver colonnes de passages
    pass_cols = [c for c in df_urg.columns if 'pass' in c.lower() or 'nbre' in c.lower()]
    print(f"\nColonnes de passages: {pass_cols}")
    
    if pass_cols and date_col_urg:
        # Agréger par date
        numeric_cols = df_urg[pass_cols].select_dtypes(include=[np.number]).columns
        if len(numeric_cols) > 0:
            df_urg_daily = df_urg.groupby(date_col_urg)[numeric_cols].sum().reset_index()
            
            # Graphique
            fig = px.line(
                df_urg_daily,
                x=date_col_urg,
                y=numeric_cols[0],
                title='🚑 Évolution des passages aux urgences',
                labels={date_col_urg: 'Date', numeric_cols[0]: 'Nombre de passages'}
            )
            fig.update_layout(template='plotly_white')
            fig.show()

## 3.3 Démographie médicale

In [ ]:
# Analyse de la démographie médicale
if datasets['medecins'] is not None:
    df_med = datasets['medecins'].copy()
    print(f"Colonnes médecins: {list(df_med.columns)}")
    display(df_med.head())
    
    # Chercher une colonne de région/département
    geo_col = None
    for col in ['departement', 'region', 'dep', 'territoire']:
        if col in df_med.columns:
            geo_col = col
            break
    
    # Chercher une colonne numérique pour les effectifs
    num_cols = df_med.select_dtypes(include=[np.number]).columns
    
    if geo_col and len(num_cols) > 0:
        # Agréger par géographie
        metric_col = num_cols[0]
        df_geo = df_med.groupby(geo_col)[metric_col].mean().sort_values(ascending=False).head(15)
        
        fig = px.bar(
            x=df_geo.values,
            y=df_geo.index.astype(str),
            orientation='h',
            title=f'👨‍⚕️ {metric_col} par {geo_col}',
            labels={'x': metric_col, 'y': geo_col},
            color=df_geo.values,
            color_continuous_scale='Blues'
        )
        fig.update_layout(template='plotly_white', showlegend=False, height=500)
        fig.show()

---
# SECTION 4 - Visualisations Pollution
---

## 4.1 Indices Airparif (Île-de-France)

In [ ]:
# Analyse des indices Airparif
if datasets['airparif'] is not None:
    df_air = datasets['airparif'].copy()
    print(f"Colonnes Airparif: {list(df_air.columns)}")
    display(df_air.head())
    
    # Statistiques
    print(f"\n📊 Statistiques Airparif:")
    display(df_air.describe())

In [ ]:
# Distribution des indices Airparif
if datasets['airparif'] is not None:
    # Chercher une colonne d'indice
    idx_cols = [c for c in df_air.columns if 'indice' in c.lower() or 'qual' in c.lower() or 'lib' in c.lower()]
    num_cols = df_air.select_dtypes(include=[np.number]).columns.tolist()
    
    if len(num_cols) >= 1:
        # Créer un histogramme pour chaque colonne numérique importante
        cols_to_plot = num_cols[:4]  # Limiter à 4 colonnes
        
        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=[f'Distribution de {c}' for c in cols_to_plot[:4]]
        )
        
        colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6']
        
        for i, col in enumerate(cols_to_plot):
            row = i // 2 + 1
            col_idx = i % 2 + 1
            fig.add_trace(
                go.Histogram(x=df_air[col].dropna(), name=col, marker_color=colors[i]),
                row=row, col=col_idx
            )
        
        fig.update_layout(
            title='🌬️ Distribution des indices Airparif',
            template='plotly_white',
            height=500,
            showlegend=False
        )
        fig.show()

## 4.2 Comparaison Paris vs Lyon vs Marseille (OpenAQ)

In [ ]:
# Combiner les données OpenAQ des 3 villes
openaq_dfs = []
for city, key in [('Paris', 'openaq_paris'), ('Lyon', 'openaq_lyon'), ('Marseille', 'openaq_marseille')]:
    if datasets.get(key) is not None:
        df_temp = datasets[key].copy()
        df_temp['city'] = city
        openaq_dfs.append(df_temp)
        print(f"{city}: {len(df_temp)} mesures")

if openaq_dfs:
    df_openaq = pd.concat(openaq_dfs, ignore_index=True)
    print(f"\n📊 Total OpenAQ: {len(df_openaq)} mesures")
    print(f"Colonnes: {list(df_openaq.columns)}")
    display(df_openaq.head())

In [ ]:
# Graphique comparatif par ville et paramètre
if openaq_dfs:
    # Chercher la colonne de paramètre (polluant) et valeur
    param_col = None
    value_col = None
    
    for col in ['parameter', 'param', 'polluant']:
        if col in df_openaq.columns:
            param_col = col
            break
    
    for col in ['last_value', 'value', 'valeur', 'moyenne']:
        if col in df_openaq.columns:
            value_col = col
            break
    
    if param_col and value_col:
        # Convertir les valeurs en numérique
        df_openaq[value_col] = pd.to_numeric(df_openaq[value_col], errors='coerce')
        
        # Box plot par polluant et ville
        fig = px.box(
            df_openaq.dropna(subset=[value_col]),
            x=param_col,
            y=value_col,
            color='city',
            title='🏙️ Comparaison des niveaux de pollution - Paris vs Lyon vs Marseille',
            labels={param_col: 'Polluant', value_col: 'Concentration', 'city': 'Ville'}
        )
        fig.update_layout(template='plotly_white', height=500)
        fig.show()
    else:
        print(f"Colonnes trouvées - param: {param_col}, value: {value_col}")

In [ ]:
# Graphique en barres groupées
if openaq_dfs and param_col and value_col:
    # Moyenne par ville et polluant
    df_summary = df_openaq.groupby(['city', param_col])[value_col].mean().reset_index()
    
    fig = px.bar(
        df_summary,
        x=param_col,
        y=value_col,
        color='city',
        barmode='group',
        title='📊 Concentration moyenne par polluant et ville',
        labels={param_col: 'Polluant', value_col: 'Concentration moyenne', 'city': 'Ville'}
    )
    fig.update_layout(template='plotly_white')
    fig.show()

---
# SECTION 5 - Qualité des Données
---

In [ ]:
def analyze_data_quality(name: str, df: pd.DataFrame):
    """Analyse la qualité d'un dataset."""
    if df is None:
        return None
    
    print(f"\n{'═'*60}")
    print(f"🔍 QUALITÉ DES DONNÉES: {name.upper()}")
    print(f"{'═'*60}")
    
    # Valeurs manquantes
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    
    missing_df = pd.DataFrame({
        'Colonne': missing.index,
        'Manquantes': missing.values,
        'Pourcentage': missing_pct.values
    })
    missing_df = missing_df[missing_df['Manquantes'] > 0].sort_values('Pourcentage', ascending=False)
    
    if len(missing_df) > 0:
        print(f"\n⚠️ Colonnes avec valeurs manquantes:")
        display(missing_df.head(10))
    else:
        print(f"\n✅ Aucune valeur manquante!")
    
    # Doublons
    n_duplicates = df.duplicated().sum()
    print(f"\n📋 Doublons: {n_duplicates:,} ({n_duplicates/len(df)*100:.2f}%)")
    
    # Valeurs négatives (pour colonnes numériques)
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    negatives = {}
    for col in numeric_cols:
        n_neg = (df[col] < 0).sum()
        if n_neg > 0:
            negatives[col] = n_neg
    
    if negatives:
        print(f"\n⚠️ Valeurs négatives détectées:")
        for col, count in negatives.items():
            print(f"   {col}: {count:,}")
    
    return {
        'missing': missing_df,
        'duplicates': n_duplicates,
        'negatives': negatives
    }

In [ ]:
# Analyser la qualité de tous les datasets
quality_reports = {}
for name, df in datasets.items():
    if df is not None:
        quality_reports[name] = analyze_data_quality(name, df)

In [ ]:
# Heatmap des valeurs manquantes
print("\n" + "="*60)
print("🗺️ HEATMAP DES VALEURS MANQUANTES")
print("="*60)

# Créer un résumé
missing_summary = []
for name, df in datasets.items():
    if df is not None:
        pct_missing = df.isnull().mean().mean() * 100
        missing_summary.append({
            'Dataset': name,
            'Taux moyen manquant (%)': round(pct_missing, 2)
        })

if missing_summary:
    ms_df = pd.DataFrame(missing_summary)
    
    fig = px.bar(
        ms_df,
        x='Dataset',
        y='Taux moyen manquant (%)',
        title='📊 Taux de valeurs manquantes par dataset',
        color='Taux moyen manquant (%)',
        color_continuous_scale='RdYlGn_r'
    )
    fig.update_layout(template='plotly_white')
    fig.show()

---
# SECTION 6 - Colonnes Clés pour le RAG
---

In [ ]:
# Documentation des colonnes clés
print("═" * 70)
print("📋 COLONNES CLÉS POUR LE RAG")
print("═" * 70)

rag_columns = {
    'SANTÉ': {
        'covid_hosp': {
            'temporelles': ['jour'],
            'géographiques': ['dep', 'sexe'],
            'métriques': ['hosp', 'rea', 'rad', 'dc'],
            'description': 'Hospitalisations COVID-19 par département et sexe'
        },
        'urgences': {
            'temporelles': ['date_de_passage'],
            'géographiques': ['dep'],
            'métriques': ['nbre_pass_corona', 'nbre_pass_tot', 'nbre_hospit_corona'],
            'description': 'Passages aux urgences pour suspicion COVID'
        },
        'medecins': {
            'temporelles': ['annee'],
            'géographiques': ['region', 'departement'],
            'métriques': ['patientele_moyenne', 'effectif'],
            'description': 'Démographie des médecins généralistes'
        }
    },
    'POLLUTION': {
        'airparif': {
            'temporelles': ['fetch_date'],
            'géographiques': ['commune', 'station'],
            'métriques': ['indice', 'no2', 'pm25', 'pm10', 'o3'],
            'description': 'Indices qualité air Île-de-France'
        },
        'openaq': {
            'temporelles': ['last_updated'],
            'géographiques': ['city', 'location'],
            'métriques': ['last_value', 'parameter'],
            'description': 'Mesures pollution temps réel (Paris, Lyon, Marseille)'
        }
    }
}

for domain, datasets_info in rag_columns.items():
    print(f"\n{'─'*60}")
    print(f"📂 {domain}")
    print(f"{'─'*60}")
    
    for ds_name, info in datasets_info.items():
        print(f"\n  📄 {ds_name}")
        print(f"     Description: {info['description']}")
        print(f"     📅 Temporelles: {info['temporelles']}")
        print(f"     🗺️  Géographiques: {info['géographiques']}")
        print(f"     📈 Métriques: {info['métriques']}")

In [ ]:
# Tableau récapitulatif des colonnes
print("\n" + "═" * 70)
print("📊 TABLEAU RÉCAPITULATIF")
print("═" * 70)

recap_data = []
for name, df in datasets.items():
    if df is not None:
        recap_data.append({
            'Dataset': name,
            'Lignes': f"{len(df):,}",
            'Colonnes': len(df.columns),
            'Col. numériques': len(df.select_dtypes(include=[np.number]).columns),
            'Col. texte': len(df.select_dtypes(include=['object']).columns),
            'Valeurs manquantes': f"{df.isnull().sum().sum():,}"
        })

recap_df = pd.DataFrame(recap_data)
display(recap_df)

---
# SECTION 7 - Insights & Recommandations
---

## 7.1 Patterns temporels identifiés

In [ ]:
print("═" * 70)
print("💡 INSIGHTS & DÉCOUVERTES")
print("═" * 70)

insights = [
    "📊 DONNÉES SANTÉ:",
    "   • Les hospitalisations COVID-19 montrent des vagues épidémiques distinctes",
    "   • Les départements urbains (75, 13, 69) sont les plus touchés",
    "   • Données disponibles depuis 2020, granularité quotidienne",
    "",
    "🌍 DONNÉES POLLUTION:",
    "   • Airparif couvre l'Île-de-France avec ~761 points de mesure",
    "   • OpenAQ fournit des mesures temps réel pour 3 grandes villes",
    "   • Polluants principaux: NO2, PM2.5, PM10, O3",
    "",
    "🔗 CORRÉLATIONS POTENTIELLES:",
    "   • Pics de pollution → augmentation passages urgences respiratoires",
    "   • Saisonnalité: pollution hiver (chauffage) vs été (ozone)",
    "   • Géographie: zones urbaines = plus de pollution + plus d'hospitalisations"
]

for line in insights:
    print(line)

## 7.2 Recommandations pour le RAG

In [ ]:
print("\n" + "═" * 70)
print("📝 RECOMMANDATIONS POUR LE RAG")
print("═" * 70)

recommendations = [
    "",
    "1️⃣  CHUNKING STRATEGY:",
    "    • Découper par département + période (semaine/mois)",
    "    • Inclure métadonnées: source, date_maj, unités",
    "    • Taille recommandée: 500-1000 tokens par chunk",
    "",
    "2️⃣  METADATA ENRICHMENT:",
    "    • Ajouter libellés des départements (ex: 75 → Paris)",
    "    • Normaliser les dates au format ISO",
    "    • Taguer le domaine: 'santé' ou 'pollution'",
    "",
    "3️⃣  QUESTIONS TYPES À SUPPORTER:",
    "    • 'Hospitalisations COVID à Paris en janvier 2024?'",
    "    • 'Niveau de NO2 à Lyon aujourd'hui?'",
    "    • 'Corrélation pollution-urgences à Marseille?'",
    "    • 'Densité médicale dans l'Hérault?'",
    "",
    "4️⃣  LIMITATIONS À DOCUMENTER:",
    "    • OpenAQ: données temps réel uniquement (pas d'historique)",
    "    • Médecins: données annuelles (pas temps réel)",
    "    • Airparif: couverture Île-de-France uniquement"
]

for line in recommendations:
    print(line)

## 7.3 Prochaines étapes

In [ ]:
print("\n" + "═" * 70)
print("🚀 PROCHAINES ÉTAPES")
print("═" * 70)

next_steps = [
    "",
    "□ 1. Nettoyer et préprocesser les données",
    "     - Gérer les valeurs manquantes",
    "     - Normaliser les formats de dates",
    "     - Enrichir avec libellés départements/régions",
    "",
    "□ 2. Créer les documents pour le RAG",
    "     - Transformer les DataFrames en texte structuré",
    "     - Appliquer la stratégie de chunking",
    "     - Ajouter les métadonnées",
    "",
    "□ 3. Implémenter la baseline (sans RAG)",
    "     - Tester GPT-3.5-turbo sur les questions types",
    "     - Mesurer les hallucinations",
    "     - Établir les métriques de référence",
    "",
    "□ 4. Implémenter RAG Basic (FAISS)",
    "     - Générer les embeddings",
    "     - Créer l'index FAISS",
    "     - Tester la récupération"
]

for line in next_steps:
    print(line)

---
## Fin du notebook d'exploration

**Auteur:** Jérôme - Master 2 Data Science  
**Date:** Février 2025  
**Projet:** OpenDataCopilot

---